# Normalización del CSV de Spotify con pandas
**Ciencia de Datos · Semana 6 · Modelamiento de datos** — CORHUILA, 2026-B

Este cuaderno toma el CSV original *Spotify Global Streaming Data (2024)* [2] (una sola tabla plana) y lo separa en tablas normalizadas hasta 3FN, siguiendo el esquema de `esquema-spotify.sql` y el diagrama `diagrama-erd-spotify.svg`. Después carga las tablas en SQLite con ese mismo DDL para comprobar que las PK, FK y restricciones se cumplen.

**Pasos:** (1) cargar y validar · (2) revisar dependencias funcionales · (3) construir dimensiones · (4) construir la tabla de hechos · (5) exportar CSV · (6) validar contra el DDL.

> Nota de transparencia: este cuaderno fue elaborado con asistencia de inteligencia artificial (Claude, Anthropic).

**Referencias**
[2] A. Soundankar, "Spotify Global Streaming Data (2024)," Kaggle, 2024.
[6] W. Kent, "A simple guide to five normal forms in relational database theory," *Commun. ACM*, vol. 26, no. 2, pp. 120–125, 1983.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

CSV_ORIGEN = Path("Cleaned_Spotify_2024_Global_Streaming_Data.csv")   # ajuste la ruta si es necesario
SQL_ESQUEMA = Path("esquema-spotify.sql")
SALIDA = Path("datos-normalizados")
SALIDA.mkdir(exist_ok=True)

pd.set_option("display.max_columns", None)

## 1. Carga y validación del CSV original
Se renombran las columnas a `snake_case` y se comprueba que el archivo tenga las columnas esperadas, sin nulos ni filas duplicadas.

In [2]:
RENOMBRAR = {
    "Country": "pais",
    "Artist": "artista",
    "Album": "album",
    "Genre": "genero",
    "Release Year": "anio_lanzamiento",
    "Monthly Listeners (Millions)": "oyentes_mensuales_m",
    "Total Streams (Millions)": "streams_totales_m",
    "Total Hours Streamed (Millions)": "horas_totales_m",
    "Avg Stream Duration (Min)": "duracion_prom_min",
    "Platform Type": "plan",
    "Streams Last 30 Days (Millions)": "streams_30d_m",
    "Skip Rate (%)": "skip_rate_pct",
}

raw = pd.read_csv(CSV_ORIGEN)
faltantes = set(RENOMBRAR) - set(raw.columns)
assert not faltantes, f"Columnas ausentes en el CSV: {faltantes}"

df = raw.rename(columns=RENOMBRAR)[list(RENOMBRAR.values())].copy()
for col in ["pais", "artista", "album", "genero", "plan"]:
    df[col] = df[col].astype(str).str.strip()

print(f"Filas: {len(df)} | Columnas: {df.shape[1]}")
print("Nulos por columna:\n", df.isna().sum().to_string())
print("Filas duplicadas:", df.duplicated().sum())
df.head()

Filas: 500 | Columnas: 12
Nulos por columna:
 pais                   0
artista                0
album                  0
genero                 0
anio_lanzamiento       0
oyentes_mensuales_m    0
streams_totales_m      0
horas_totales_m        0
duracion_prom_min      0
plan                   0
streams_30d_m          0
skip_rate_pct          0
Filas duplicadas: 0


,pais,artista,album,genero,anio_lanzamiento,oyentes_mensuales_m,streams_totales_m,horas_totales_m,duracion_prom_min,plan,streams_30d_m,skip_rate_pct
0,Germany,Taylor Swift,1989 (Taylor's Version),K-pop,2020,23.10,3695.53,14240.35,4.28,Free,118.51,2.24
1,Brazil,The Weeknd,After Hours,R&B,2023,60.60,2828.16,11120.44,3.90,Premium,44.87,23.98
2,United States,Post Malone,Austin,Reggaeton,2019,42.84,1425.46,4177.49,4.03,Free,19.46,4.77
3,Italy,Ed Sheeran,Autumn Variations,K-pop,2019,73.24,2704.33,12024.08,3.26,Premium,166.05,25.12
4,Italy,Ed Sheeran,Autumn Variations,R&B,2019,7.89,3323.25,13446.32,4.47,Free,173.43,15.82


## 2. Dependencias funcionales (base de la normalización)
Antes de separar, se comprueba qué atributo determina a cuál. Si `X → Y` se cumple, `Y` pertenece a la tabla cuya clave es `X`; guardarlo en otra parte lo repetiría (anomalía de actualización) [6].

In [3]:
def determina(df, x, y):
    """True si cada valor de x tiene un único valor de y (x -> y)."""
    return bool((df.groupby(x)[y].nunique(dropna=False) <= 1).all())

pruebas = [
    (["artista", "album", "anio_lanzamiento"], ["genero"]),
    (["artista", "album"], ["anio_lanzamiento"]),
    (["album"], ["artista"]),
    (["pais", "artista", "album", "plan"], ["streams_totales_m"]),
]
for x, y in pruebas:
    print(f"{' + '.join(x):<38} -> {' + '.join(y):<20} {determina(df, x, y[0])}")

artista + album + anio_lanzamiento     -> genero               False
artista + album                        -> anio_lanzamiento     True
album                                  -> artista              True
pais + artista + album + plan          -> streams_totales_m    False


Lectura: lo que resulte `True` en las dos primeras filas indica atributos que dependen del álbum y no deben repetirse en la tabla de hechos. Lo que resulte `False` se trata como atributo de la observación país × álbum × plan: por eso `genero_id` se guarda en `metrica_streaming` y no en `album`. El diseño es válido en ambos casos.

## 3. Dimensiones (1FN → 3FN): una tabla por concepto

In [4]:
# --- pais (con código ISO y región; el CSV solo trae el nombre) -------------
GEO = {
    "USA": ("US", "Norteamérica"), "United States": ("US", "Norteamérica"),
    "Canada": ("CA", "Norteamérica"), "Mexico": ("MX", "Latinoamérica"),
    "Brazil": ("BR", "Latinoamérica"), "Argentina": ("AR", "Latinoamérica"),
    "Colombia": ("CO", "Latinoamérica"), "Chile": ("CL", "Latinoamérica"),
    "UK": ("GB", "Europa"), "United Kingdom": ("GB", "Europa"),
    "Germany": ("DE", "Europa"), "France": ("FR", "Europa"),
    "Spain": ("ES", "Europa"), "Italy": ("IT", "Europa"),
    "Sweden": ("SE", "Europa"), "Netherlands": ("NL", "Europa"),
    "India": ("IN", "Asia"), "Japan": ("JP", "Asia"),
    "South Korea": ("KR", "Asia"), "Indonesia": ("ID", "Asia"),
    "Australia": ("AU", "Oceanía"), "New Zealand": ("NZ", "Oceanía"),
    "South Africa": ("ZA", "África"), "Nigeria": ("NG", "África"),
    "Russia": ("RU", "Europa"), "Turkey": ("TR", "Asia"),
}
sin_mapeo = sorted(set(df["pais"]) - set(GEO))
assert not sin_mapeo, f"Agregue estos países al diccionario GEO: {sin_mapeo}"

pais = (pd.DataFrame({"nombre": sorted(df["pais"].unique())})
          .assign(country_id=lambda t: range(1, len(t) + 1),
                  codigo_iso=lambda t: t["nombre"].map(lambda n: GEO[n][0]),
                  region=lambda t: t["nombre"].map(lambda n: GEO[n][1]))
          [["country_id", "codigo_iso", "nombre", "region"]])

# --- genero y plan_suscripcion (catálogos pequeños) --------------------------
genero = (pd.DataFrame({"nombre": sorted(df["genero"].unique())})
            .assign(genero_id=lambda t: range(1, len(t) + 1))[["genero_id", "nombre"]])
plan = (pd.DataFrame({"nombre": sorted(df["plan"].unique())})
          .assign(plan_id=lambda t: range(1, len(t) + 1))[["plan_id", "nombre"]])

# --- artista -----------------------------------------------------------------
artista = (pd.DataFrame({"nombre": sorted(df["artista"].unique())})
             .assign(artist_id=lambda t: range(1, len(t) + 1),
                     genero_principal=None)[["artist_id", "nombre", "genero_principal"]])

# --- album (FK -> artista) ---------------------------------------------------
album = (df[["artista", "album", "anio_lanzamiento"]].drop_duplicates()
           .merge(artista[["artist_id", "nombre"]], left_on="artista", right_on="nombre")
           .rename(columns={"album": "titulo"})
           .sort_values(["artist_id", "titulo", "anio_lanzamiento"])
           .reset_index(drop=True))
album["album_id"] = album.index + 1
album = album[["album_id", "artist_id", "titulo", "anio_lanzamiento"]]

for nombre, t in [("pais", pais), ("genero", genero), ("plan_suscripcion", plan),
                  ("artista", artista), ("album", album)]:
    print(f"{nombre}: {len(t)} filas")
pais.head()

pais: 20 filas
genero: 10 filas
plan_suscripcion: 2 filas
artista: 15 filas
album: 15 filas


,country_id,codigo_iso,nombre,region
0,1,AR,Argentina,Latinoamérica
1,2,AU,Australia,Oceanía
2,3,BR,Brazil,Latinoamérica
3,4,CA,Canada,Norteamérica
4,5,FR,France,Europa


## 4. Tabla de hechos `metrica_streaming`
Se reemplazan los textos repetidos (país, álbum, género, plan) por sus claves foráneas. Solo quedan las medidas numéricas.

In [5]:
clave_album = df.merge(artista[["artist_id", "nombre"]], left_on="artista", right_on="nombre") \
                .merge(album, left_on=["artist_id", "album", "anio_lanzamiento"],
                       right_on=["artist_id", "titulo", "anio_lanzamiento"])

hechos = (clave_album
          .merge(pais[["country_id", "nombre"]].rename(columns={"nombre": "pais"}), on="pais")
          .merge(genero.rename(columns={"nombre": "genero"}), on="genero")
          .merge(plan.rename(columns={"nombre": "plan"}), on="plan"))

assert len(hechos) == len(df), "El cruce con las dimensiones cambió el número de filas"

medidas = ["oyentes_mensuales_m", "streams_totales_m", "horas_totales_m",
           "duracion_prom_min", "streams_30d_m", "skip_rate_pct"]
metrica = (hechos[["country_id", "album_id", "genero_id", "plan_id"] + medidas]
           .sort_values(["country_id", "album_id", "plan_id"]).reset_index(drop=True))
metrica.insert(0, "metric_id", metrica.index + 1)
metrica.head()

,metric_id,country_id,album_id,genero_id,plan_id,oyentes_mensuales_m,streams_totales_m,horas_totales_m,duracion_prom_min,streams_30d_m,skip_rate_pct
0,1,1,1,6,1,27.47,2014.64,6539.73,3.31,198.06,9.81
1,2,1,1,10,2,82.55,4906.61,12352.92,3.84,48.83,26.05
2,3,1,1,7,2,65.27,2634.12,8536.54,2.59,46.67,32.29
3,4,1,2,9,2,18.90,3479.59,13046.66,4.31,9.69,15.21
4,5,1,2,1,2,59.11,1772.23,7427.48,3.73,183.82,24.02


## 5. Exportar un CSV por tabla

In [6]:
tablas = {"pais": pais, "genero": genero, "plan_suscripcion": plan,
          "artista": artista, "album": album, "metrica_streaming": metrica}
for nombre, t in tablas.items():
    t.to_csv(SALIDA / f"{nombre}.csv", index=False, encoding="utf-8")
    print(f"{nombre}.csv  ->  {len(t)} filas")

pais.csv  ->  20 filas
genero.csv  ->  10 filas
plan_suscripcion.csv  ->  2 filas
artista.csv  ->  15 filas
album.csv  ->  15 filas
metrica_streaming.csv  ->  500 filas


## 6. Validación contra el DDL
Se ejecuta `esquema-spotify.sql` en una base SQLite en memoria, se cargan los CSV y se verifica que no se viole ninguna PK, FK ni `CHECK`. Si algo falla, SQLite lanza el error.

In [7]:
con = sqlite3.connect(":memory:")
con.executescript(SQL_ESQUEMA.read_text(encoding="utf-8"))

orden_carga = ["pais", "genero", "plan_suscripcion", "artista", "album", "metrica_streaming"]
for nombre in orden_carga:                       # padres antes que hijos
    pd.read_csv(SALIDA / f"{nombre}.csv").to_sql(nombre, con, if_exists="append", index=False)

violaciones = con.execute("PRAGMA foreign_key_check").fetchall()
print("Violaciones de FK:", violaciones if violaciones else "ninguna")
for nombre in orden_carga:
    print(f"{nombre:<20}", con.execute(f"SELECT COUNT(*) FROM {nombre}").fetchone()[0], "filas")

Violaciones de FK: ninguna
pais                 20 filas
genero               10 filas
plan_suscripcion     2 filas
artista              15 filas
album                15 filas
metrica_streaming    500 filas


**Consulta de comprobación:** horas transmitidas por país (el mismo tipo de pregunta que motiva la decisión de capacidad del caso).

In [8]:
pd.read_sql_query("""
    SELECT p.nombre AS pais, ROUND(SUM(m.horas_totales_m), 1) AS horas_millones
    FROM metrica_streaming m
    JOIN pais p ON p.country_id = m.country_id
    GROUP BY p.nombre
    ORDER BY horas_millones DESC
""", con)

,pais,horas_millones
0,South Korea,315992.4
1,Sweden,315891.9
2,South Africa,296353.8
3,Brazil,262069.8
4,Russia,261481.6
5,Italy,255918.5
6,Mexico,249586.9
7,France,245512.1
8,Netherlands,220382.8
9,Argentina,214954.7
